In [17]:
import json
from pathlib import Path

import pandas as pd
import plotly.express as px
from transformers import PreTrainedTokenizer

from pivotal_tokens.constants import get_data_dir, get_artifacts_dir
from pivotal_tokens.hf.loading import  load_tokenizer
from pivotal_tokens.repo import SampleRepo, DictRepo


EXP_DIR = get_data_dir() / 'experiments' / 'exp2.1-sps-tokens' / 'exp2.1.2-qwen3-1.7b-sps-tokens-small-prob-threshold'
# EXP_DIR = get_data_dir() / 'experiments' / 'exp2.1-sps-tokens' / 'exp2.1.3-qwen3-1.7b-sps-tokens-small-prob-threshold'
# EXP_DIR = get_data_dir() / 'experiments' / 'exp2.1-sps-tokens' / 'exp2.1.4-qwen3-1.7b-sps-tokens-small-prob-threshold'
REPO_DIR = EXP_DIR / 'data' / 'repo'
PIVOTAL_TOKENS_FILE = EXP_DIR / 'data' / 'pivotal_tokens.csv'

TRACES_FILE = get_artifacts_dir() / 'exp1.1.1-qwen3-1.7b-traces.csv'
PIVOTAL_TOKENS_FILE = get_artifacts_dir() / 'exp2.1.2-qwen3-1.7b-pivotal-tokens.csv'

QWEN3_1_7B_MODEL_ID = 'Qwen/Qwen3-1.7B'

In [11]:
tokenizer = load_tokenizer(QWEN3_1_7B_MODEL_ID)

In [12]:
base_repo = DictRepo(dirpath=REPO_DIR)

In [14]:
traces_df = pd.read_csv(TRACES_FILE)
traces_df[:2]

,id,query,ground_truth,trace
0,5a8b57f25542995d1e6f1371,Were Scott Derrickson and Ed Wood of the same ...,yes,"<think>\nOkay, let's see. The question is whet..."
1,5a8db19d5542994ba4e3dd00,Are Local H and For Against both from the Unit...,yes,"<think>\nOkay, let's see. The question is aski..."


In [15]:
tokens_df = pd.read_csv(PIVOTAL_TOKENS_FILE)
tokens_df['token_ids'] = tokens_df['token_ids'].apply(json.loads)
tokens_df[:2]

,sample_id,span_id,token_ids,span_text,prob_before,prob_after,prob_delta,is_pivotal,pivotal_context,metadata
0,5a8db19d5542994ba4e3dd00,5cac3872-e49d-11f0-93c4-08bfb8a7cb42,[220],,0.58,0.64,0.06,True,<|im_start|>system\nAnswer the following quest...,"{'id': '5a8db19d5542994ba4e3dd00', 'question':..."
1,5a8db19d5542994ba4e3dd00,5caca596-e49d-11f0-93c4-08bfb8a7cb42,[2704],sure,0.62,0.54,-0.08,True,<|im_start|>system\nAnswer the following quest...,"{'id': '5a8db19d5542994ba4e3dd00', 'question':..."


In [16]:
df = pd.merge(traces_df, tokens_df, left_on='id', right_on='sample_id', how='inner')
df[:2]

,id,query,ground_truth,trace,sample_id,span_id,token_ids,span_text,prob_before,prob_after,prob_delta,is_pivotal,pivotal_context,metadata
0,5a8db19d5542994ba4e3dd00,Are Local H and For Against both from the Unit...,yes,"<think>\nOkay, let's see. The question is aski...",5a8db19d5542994ba4e3dd00,5cac3872-e49d-11f0-93c4-08bfb8a7cb42,[220],,0.58,0.64,0.06,True,<|im_start|>system\nAnswer the following quest...,"{'id': '5a8db19d5542994ba4e3dd00', 'question':..."
1,5a8db19d5542994ba4e3dd00,Are Local H and For Against both from the Unit...,yes,"<think>\nOkay, let's see. The question is aski...",5a8db19d5542994ba4e3dd00,5caca596-e49d-11f0-93c4-08bfb8a7cb42,[2704],sure,0.62,0.54,-0.08,True,<|im_start|>system\nAnswer the following quest...,"{'id': '5a8db19d5542994ba4e3dd00', 'question':..."


In [18]:
df.to_csv(PIVOTAL_TOKENS_FILE, index=False)